In [1]:
import sys
print(sys.prefix)

C:\Users\mosab\anaconda3\envs\tf


In [2]:
#!pip install bitsandbytes

In [3]:
#!pip install accelerate

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Check if a GPU is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigscience/bloomz-7b1")

# Load the model and quantize it to 4-bit
model = AutoModelForCausalLM.from_pretrained("bigscience/bloomz-7b1", 
                                             #device_map='auto', 
                                             load_in_4bit=True,
                                             torch_dtype=torch.float16,
                                             bnb_4bit_compute_dtype=torch.float16  # Set the compute data type to float16 for better performance
                                            )#.to(device)

# Initialize the text generation pipeline with the 4-bit model
#generator = pipeline('text-generation', model=model, tokenizer=tokenizer)#, device=0 if device == 'cuda' else -1)

# Generate text
#prompt = "Once upon a time"
#result = generator(prompt, max_length=50, num_return_sequences=1)

# Print the generated text
#print(result[0]['generated_text'])


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


In [5]:
import pandas as pd
# Load the ARCD dataset from CSV
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

In [6]:
df1=df

In [7]:
df1.shape

(1395, 5)

In [8]:
i=0

In [9]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import re


# Read stopwords from a file (assuming each stopword is on a new line)
#stopwords_list = read_stopwords(r'C:\Users\mosab\Desktop\Jupyter Notebook\Research\ArabicStopwords.txt')

# Function to read stopwords from a file and return them as a list
# def read_stopwords(file_path):
#     with open(file_path, 'r', encoding="utf8") as file:
#         stopwords = [line.strip() for line in file]
#     return stopwords



#df=df1[i:i+50]
#i+=50
# Function to extract answers using BLOOMZ-7B1
def extract_answers_from_context(question, context, title):
    # Combine title, context, and question to create the prompt
    prompt = f"Title: {title}\nContext: {context}\nInstruction: When there is no answer, say 'No answer.' only.\nQuestion: {question}\nAnswer:"
    
    # Tokenize input and generate output using the model
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(inputs['input_ids'], max_new_tokens=100, do_sample=False, pad_token_id=tokenizer.eos_token_id)#max_length=356
    
    # Decode the generated output
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract the answer from the generated text
    answer = generated_text.split("Answer:")[-1].strip()
    return answer

# To match only Arabic letters in Regex, we use unicode:

# [\u0621-\u064A]+
# Or we can simply use Arabic letters directly

# [ء-ي]+

# Text normalization for comparison (lowercase, remove articles, punctuation, etc.)
def normalize_text(text):
    text = text.lower()
    #text = re.sub(r'\b(a|an|the)\b', ' ', text)  # Remove articles
    text = re.sub(r'[\u064B-\u065F]', '', text)  # Remove diacritics
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation and double whitespaces
    #text = re.sub(r'[^a-zء-ي0-9]+', ' ', text)  # Remove double whitespaces
    #text = ' '.join(text.split())  # Remove extra spaces
    return text

# Exact Match (EM) calculation
def exact_match_score(predicted_answer, actual_answer):
    return int(normalize_text(predicted_answer) == normalize_text(actual_answer))

# F1 Score calculation for a single prediction
def f1_single(predicted_answer, actual_answer):
    # Tokenize predicted and actual answers
    pred_tokens = normalize_text(predicted_answer).split()
    actual_tokens = normalize_text(actual_answer).split()
    
    common_tokens = set(pred_tokens) & set(actual_tokens)
    #common_tokens = remove_stopwords_from_set(common_tokens, stopwords_list)
    if len(common_tokens) == 0:
        return 0.0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(actual_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Function to calculate average EM and F1 scores across the dataset
def evaluate_qa(predicted_answers, actual_answers):
    total_em = 0
    total_f1 = 0
    n = len(predicted_answers)
    
    for predicted, actual in zip(predicted_answers, actual_answers):
        total_em += exact_match_score(predicted, actual)
        total_f1 += f1_single(predicted, actual)
    
    # Calculate the average EM and F1 scores
    em_score = total_em / n
    f1_score_avg = total_f1 / n
    return em_score, f1_score_avg


# Function to compute exact match (EM) and F1 score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Strip white space in predictions and labels
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    exact_match_score, avg_f1_score = evaluate_qa(decoded_preds, decoded_labels)
    
    # Exact Match
    #exact_matches = [1 if pred == label else 0 for pred, label in zip(decoded_preds, decoded_labels)]
    #exact_match_score = sum(exact_matches) / len(exact_matches)
    
    # F1 Score
    #f1_scores = [f1_metric.compute(predictions=[pred], references=[label])['f1'] for pred, label in zip(decoded_preds, decoded_labels)]
    #avg_f1_score = sum(f1_scores) / len(f1_scores)
    
    
    return {
        "exact_match": exact_match_score,
        "f1": avg_f1_score
    }


In [21]:
questions=[]
contexts=[]
# List to store predicted and actual answers
predicted_answers = []
actual_answers = []

# Iterate over the dataset and extract answers for each row
for idx, row in df.iterrows():
    question = row["question"]
    context = row["context"]
    title = row["title"]
    
    questions.append(question)
    contexts.append(context)
    # Extract answer from the model
    predicted_answer = extract_answers_from_context(question, context, title)
    predicted_answers.append(predicted_answer)
    
    # Get the actual answer(s) from the dataset
    actual_answer = row["answer"]
    actual_answers.append(actual_answer)

# Evaluate and print confusion matrix, accuracy, EM, and F1 scores
y_true = [1 if actual in predicted else 0 for predicted, actual in zip(predicted_answers, actual_answers)]
y_pred = [1 if predicted in actual else 0 for predicted, actual in zip(predicted_answers, actual_answers)]

# Confusion matrix with labels
labels = ["Incorrect", "Correct"]

# Classification report
#print("\nClassification Report:")
#print(classification_report(y_true, y_pred, target_names=labels))

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nAccuracy: {accuracy:.2f}")

# Calculate Exact Match and F1 Score
em_score, f1_score_avg = evaluate_qa(predicted_answers, actual_answers)
print(f"\nExact Match Score (EM): {em_score:.2f}")
print(f"F1 Score: {f1_score_avg:.2f}")


Accuracy: 0.26

Exact Match Score (EM): 0.31
F1 Score: 0.64


In [14]:
print(len(y_true))

50


In [15]:
print(len(y_pred))

50


In [16]:
print(len(predicted_answers))

50


In [17]:
print(len(actual_answers))
i=0

50


In [22]:
for context, question, actual_answer, predicted_answer in list(zip(contexts,questions, actual_answers, predicted_answers))[100:130]:
    print(context)
    print(question)
    print(actual_answer)
    print(predicted_answer)
    print("=============================================")
i+=30  

العِرَاقُ أو (رسمياً: جُمْهُوريَّة العِرَاق)، و(بالكردية: كۆماری عێراق)، هي دولة عربية، وتعد جمهورية برلمانية اتحادية وفقاً لدستور العراق، وتتكون من ثمان عشرة محافظة رسمياً، وتسع عشرة محافظة بحكم الأمر الواقع (محافظة حلبجة)، عاصمته بغداد، والعراق أحد دول غرب آسيا المطلة على الخليج العربي.
على ماذا تُطِل العراق؟
['الخليج العربي.']
الخليج العربي
العِرَاقُ أو (رسمياً: جُمْهُوريَّة العِرَاق)، و(بالكردية: كۆماری عێراق)، هي دولة عربية، وتعد جمهورية برلمانية اتحادية وفقاً لدستور العراق، وتتكون من ثمان عشرة محافظة رسمياً، وتسع عشرة محافظة بحكم الأمر الواقع (محافظة حلبجة)، عاصمته بغداد، والعراق أحد دول غرب آسيا المطلة على الخليج العربي.
من كم محافظة تتكون دولة العِراق؟
['ثمان عشرة محافظة']
ثمان عشرة
العِرَاقُ (رسمياً: جُمْهُوريَّة العِرَاق)، و(بالكردية: كۆماری عێراق)، هو جمهورية برلمانية اتحادية وفقاً لدستور العراق، ويتكون من ثمان عشرة محافظة رسمياً، وتسع عشرة محافظة بحكم الأمر الواقع (محافظة حلبجة)، عاصمته بغداد، والعراق أحد دول غرب آسيا المطلة على الخليج العربي. يحده من الجنوب الكويت والمملكة

In [29]:
max_len=0
a=""
for actual_answer in actual_answers:
    if(len(actual_answer)>max_len):
        max_len= len(actual_answer)
        a=actual_answer
        
print(max_len, "\n", a)

125 
 ['المصريين القدماء، الآشوريين، الفرس، الإغريق، الرومان، الروم البيزنطيين، العرب، الصليبيين، الأتراك العثمانيين، فالفرنسيين.']


In [28]:
max_len=0
c=""
q=""
a=""
p=""
for context, question, actual_answer, predicted_answer in list(zip(contexts,questions, actual_answers, predicted_answers)):
    
    if((len(context)+len(question)+len(actual_answer)+len(predicted_answer))>max_len):
        max_len=len(context)+len(question)+len(actual_answer)+len(predicted_answer)
        c=context
        q=question
        a=actual_answer
        p=predicted_answer  
        
print(max_len, "\n", c, "\n", q, "\n", a, "\n", p)

تكررت الإنقلابات العسكرية في تاريخ السودان الحديث، وفي عام 1989 م، قاد العميد عمر البشير انقلاباً عسكرياً، أطاح بحكومة مدنية برئاسة الصادق المهدي زعيم ، وأصبح رئيساً لمجلس قيادة ثورة الإنقاذ، ثم رئيساً للجمهورية إلى الآن.
من كان رئيس الجكومة المدنية وقت الإنقلاب؟
['الصادق المهدي']
الصادق المهدي
2253 
 قُسِّمت الأندلس إلى خمس وحداتٍ إداريَّة بعد فتحها واستقرار الحُكم الإسلامي فيها، وتلك الوحدات تُقابلُ تقريبًا كُلًا من: منطقة أندلوسيا، والجُمهوريَّة الپرتغاليَّة، ومنطقة جليقية (غاليسيا)، ومنطقة أراگون المُعاصرة؛ ومنطقة قشتالة، ومملكة ليون، وكونتيَّة برشلونة، ومنطقة سپتمانيا التاريخيَّة. أمَّا من الناحية السياسيَّة، فقد كانت في بادئ الأمر تُشكِّلُ ولايةً من ولايات الدولة الأُمويَّة زمن الخليفة الوليد بن عبد الملك، وبعد انهيار الدولة الأُمويَّة وقيام الدولة العبَّاسيَّة، استقلَّ عبد الرحمٰن بن مُعاوية، وهو أحد أُمراء بني أُميَّة الناجين من سُيُوف العبَّاسيين، استقلَّ بالأندلس وأسس فيها إمارة قُرطُبة، فدامت 179 سنة، وقام بعدها عبد الرحمٰن الناصر لِدين الله بإعلان الخِلافة الأُمويَّة عوض ال

In [10]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import BitsAndBytesConfig
#import evaluate

# 1. Prepare the dataset (Assumes the CSV has context, title, question, and answers)
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')
df_train = df[0:int(df.shape[0]*0.8)]
df_val = df[int(df.shape[0]*0.8)+1:int(df.shape[0]*0.9)]
df_test = df[int(df.shape[0]*0.9)+1:df.shape[0]]#-1

# 2. Function to create prompt (make sure this returns a dictionary)
def create_prompt(row):
    title = row['title']
    context = row['context']
    question = row['question']
    prompt = f"Title: {title}\nContext: {context}\nQuestion: {question}\nAnswer:"
    return {'input': prompt, 'output': row['answer']}  # Return dictionary

# 3. Apply the prompt creation function to each dataset
train_prompts = df_train.apply(create_prompt, axis=1).tolist()  # Convert to list of dictionaries
val_prompts = df_val.apply(create_prompt, axis=1).tolist()
test_prompts = df_test.apply(create_prompt, axis=1).tolist()

# 4. Convert to pandas DataFrame to ensure compatibility with Dataset
train_df = pd.DataFrame(train_prompts)
val_df = pd.DataFrame(val_prompts)
test_df = pd.DataFrame(test_prompts)

# 5. Convert the DataFrame to HuggingFace Dataset
train_hf_dataset = Dataset.from_pandas(train_df)
val_hf_dataset = Dataset.from_pandas(val_df)
test_hf_dataset = Dataset.from_pandas(test_df)



In [ ]:

# 5. Load the tokenizer and the model with 4-bit precision
#model_name = "bigscience/bloomz-7b1"

# Enable 4-bit precision to reduce memory usage
#bnb_config = BitsAndBytesConfig(load_in_4bit=True)

#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)

# 6. Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples['input'], padding="max_length", truncation=True, max_length=2400, return_tensors="pt")
    labels = tokenizer(examples['output'], padding="max_length", truncation=True, max_length=200, return_tensors="pt").input_ids
    inputs['labels'] = labels
    return inputs

# 7. Tokenize the datasets
train_tokenized = train_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])
val_tokenized = val_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])
test_tokenized = test_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# 8. Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./bloomz_finetuned_qa_4bit",
    eval_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=2,  # Adjust based on your GPU memory
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=500,
    fp16=True  # Mixed precision training for more efficiency
)

# 9. Exact Match and F1 Score Metrics
# Load evaluation metrics
#em_metric = evaluate.load("exact_match")
#f1_metric = evaluate.load("f1")

# 10. Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

# 11. Train the model
trainer.train()

# 12. Evaluate on validation and test datasets
print("Evaluating on Validation Set")
validation_results = trainer.evaluate(eval_dataset=val_tokenized)
print(f"Validation Results: {validation_results}")

print("Evaluating on Test Set")
test_results = trainer.evaluate(eval_dataset=test_tokenized)
print(f"Test Results: {test_results}")


In [10]:
from datetime import datetime
now = datetime.now()
print(now.time())

12:14:35.490126


In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from datasets import Dataset
import pandas as pd

# Load tokenizer
#3model_name = "bigscience/bloomz-7b1"
#tokenizer = AutoTokenizer.from_pretrained(model_name)

# 1. Load the quantized model with 4-bit precision
#model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto")

# 2. Set up the PEFT configuration (LoRA)
config = LoraConfig(
    r=16,                  # LoRA attention dimension
    lora_alpha=32,          # LoRA scaling factor
    lora_dropout=0.1,       # Dropout rate
    bias="none",            # Optional: Specify which layers to apply LoRA
    task_type="CAUSAL_LM"   # Type of task (causal language modeling)
)

# 3. Apply PEFT to the model
model = get_peft_model(model, config)

# Check model parameters
model.print_trainable_parameters()

# 1. Prepare the dataset (Assumes the CSV has context, title, question, and answers)
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')
df_train = df[0:int(df.shape[0]*0.8)]
df_val = df[int(df.shape[0]*0.8)+1:int(df.shape[0]*0.9)]
df_test = df[int(df.shape[0]*0.9)+1:df.shape[0]]#-1

# Function to create prompt (ensure it returns a dictionary)
def create_prompt(row):
    title = row['title']
    context = row['context']
    question = row['question']
    prompt = f"Title: {title}\nContext: {context}\nQuestion: {question}\nAnswer:"
    return {'input': prompt, 'output': row['answer']}

train_prompts = df_train.apply(create_prompt, axis=1).tolist()
val_prompts = df_val.apply(create_prompt, axis=1).tolist()
test_prompts = df_test.apply(create_prompt, axis=1).tolist()

# Convert to DataFrame and then HuggingFace Dataset
train_df = pd.DataFrame(train_prompts)
val_df = pd.DataFrame(val_prompts)
test_df = pd.DataFrame(test_prompts)

train_hf_dataset = Dataset.from_pandas(train_df)
val_hf_dataset = Dataset.from_pandas(val_df)
test_hf_dataset = Dataset.from_pandas(test_df)

# 5. Tokenization
def preprocess_function(examples):
    inputs = examples['input']
    outputs = examples['output']
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=512)
    labels = tokenizer(outputs, padding="max_length", truncation=True, max_length=512)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Tokenize datasets
train_tokenized = train_hf_dataset.map(preprocess_function, batched=True)
val_tokenized = val_hf_dataset.map(preprocess_function, batched=True)
test_tokenized = test_hf_dataset.map(preprocess_function, batched=True)

# 6. Set up training arguments and train the model
from transformers import Trainer, TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=8,
#     num_train_epochs=3,
#     logging_dir="./logs",
#     logging_steps=10,
#     save_strategy="epoch",  # Change save strategy to "epoch"
#     eval_strategy="epoch",  # Match the evaluation strategy
#     load_best_model_at_end=True,  # Now this will work
#     report_to="none",  # Avoids logging to external platforms like WandB
# )


# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=8,
#     num_train_epochs=3,
#     logging_dir="./logs",
#     logging_steps=10,
#     save_steps=500,  # Save every 500 steps (for example)
#     evaluation_strategy="steps",  # Match the evaluation strategy to "steps"
#     eval_steps=500,  # Evaluate every 500 steps
#     save_strategy="steps",  # Save at the same interval
#     load_best_model_at_end=True,  # This will now work
#     report_to="none",  # Avoids logging to external platforms like WandB
# )

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,  # Reduced batch size
    per_device_eval_batch_size=2,   # Reduced evaluation batch size
    gradient_accumulation_steps=16,  # Accumulate gradients over multiple steps
    fp16=True,  # Enable mixed precision training
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    # Optionally, use CPU if GPU is not feasible
    # no_cuda=True,  # Uncomment to use CPU
)

# Optionally, clear cache if running out of memory
import torch
torch.cuda.empty_cache()



# 7. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 8. Fine-tune the model
trainer.train()

# 9. Evaluate the model on the validation set
val_results = trainer.evaluate(eval_dataset=val_tokenized)
print(f"Validation Results: {val_results}")

# 10. Evaluate the model on the test dataset
test_results = trainer.evaluate(eval_dataset=test_tokenized)  # Test dataset evaluation
print(f"Test Results: {test_results}")

trainable params: 7,864,320 || all params: 7,076,880,384 || trainable%: 0.1111


Map:   0%|          | 0/1116 [00:00<?, ? examples/s]

Map:   0%|          | 0/138 [00:00<?, ? examples/s]

Map:   0%|          | 0/139 [00:00<?, ? examples/s]

C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
0,14.109400,13.646808
1,11.531500,10.020324
2,8.593800,8.400101


C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mosab\anaconda3\envs\tf\lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Validation Results: {'eval_loss': 8.400100708007812, 'eval_runtime': 2403.9111, 'eval_samples_per_second': 0.057, 'eval_steps_per_second': 0.029, 'epoch': 2.924731182795699}
Test Results: {'eval_loss': 8.349581718444824, 'eval_runtime': 2421.2439, 'eval_samples_per_second': 0.057, 'eval_steps_per_second': 0.029, 'epoch': 2.924731182795699}


In [12]:
now = datetime.now()
print(now.time())

12:19:14.815110
